In [ ]:
from PIL import Image
import numpy as np
import cv2
import base64
import io

import onnxruntime as ort

np.set_printoptions(
    threshold=np.inf,
    linewidth=np.inf,
    precision=2,
    suppress=True
)

def resize_image(img: Image):
    '''
    Transforms a given image of an unsolved equation (eg. an image of 2 + 3) and resizes it down for further processing.

    Parameters:
    image_location: The location of the image to be processed.

    Returns:
    A scaled down numpy array of the image.
    '''

    # FIXME: Function should eventually be reformed to transform images that have more than three symbols.

    # Load and preprocess the image.
    img = img.convert('L')

    img_width, img_height = img.size

    if img_width > img_height:
        img = img.resize((int(img_width / img_height * 28), 28))
    else:
        img = img.resize((28, int(img_height / img_width * 28)))

    # Inverts colors and filters out any subtle dark colors.
    img_array = 255 - np.array(img)     
    img_array[img_array < 10] = 0

    return img_array

def find_contours(image: Image):
    # Finds the contours by obtaining only the outermost shapes.
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (5, 3))
    morphed = cv2.morphologyEx(image, cv2.MORPH_CLOSE, kernel)

    contours, _ = cv2.findContours(morphed, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    boxes = sorted([cv2.boundingRect(c) for c in contours], key=lambda b: b[0])

    print(type(boxes))

    return boxes

with open("data/3+3.txt", "r") as file:
    image = Image.open(io.BytesIO(base64.decodebytes(bytes(file.read(), "utf-8"))))

image = resize_image(image)
rect_boxes = find_contours(image)

print(image)

In [ ]:
def harsh_gamma_enhance(data, gamma=0.3, target_max=255):
    """
    Very aggressive gamma correction - makes mid-tones much brighter
    """
    nonzero_mask = data > 1e-6
    result = data.copy().astype(float)
    
    if np.any(nonzero_mask):
        nonzero_vals = data[nonzero_mask]
        # Normalize to [0,1]
        normalized = (nonzero_vals - nonzero_vals.min()) / (nonzero_vals.max() - nonzero_vals.min())
        
        # Apply harsh gamma (0.3 is very aggressive)
        gamma_corrected = np.power(normalized, gamma)
        
        result[nonzero_mask] = gamma_corrected * target_max
    
    result[data <= 1e-6] = 0
    return result.astype(np.uint8)

def stretch_image(image: np.ndarray, rect_boxes: list) -> np.ndarray:
    images = np.ndarray((len(rect_boxes), 1, 28, 28))

    for i, rect in enumerate(rect_boxes):
        x, y, w, h = rect

        curr_image = image[y:y + h, x:x + w]

        # Finds the amount of padding required to make the image a square.
        if w > h:
            pad_h = w - h

            top = pad_h // 2
            bottom = pad_h - top

            left = right = 0
        elif h > w:
            pad_w = h - w

            left = pad_w // 2
            right = pad_w - left
            
            top = bottom = 0
        else:
            top = bottom = left = right = 0

        padded_image = cv2.copyMakeBorder(curr_image, top, bottom, left, right, cv2.BORDER_CONSTANT, 0)
        padded_image = cv2.resize(padded_image, (28, 28), interpolation=cv2.INTER_CUBIC)

        images[i] = padded_image

    return images

def normalize(image: np.ndarray) -> np.ndarray:
    return image / 255

In [ ]:
images = np.ndarray((len(rect_boxes), 1, 28, 28))

# adds padding to each image
for i, rect in enumerate(rect_boxes):
    x, y, w, h = rect

    print(1)

    curr_image = image[y:y + h, x:x + w]

    # pad_w = 28 - w
    # pad_h = 28 - h

    # left = pad_w // 2
    # right = pad_w - left

    # top = pad_h // 2
    # bottom = pad_h - top

    # padded_image = cv2.copyMakeBorder(curr_image, top, bottom, left, right, cv2.BORDER_CONSTANT, 0)

    # images[i] = padded_image

    # print(curr_image.shape)

    if w > h:
        pad_h = w - h

        top = pad_h // 2
        bottom = pad_h - top

        left = right = 0

        print(left, right, top, bottom)
    elif h > w:
        pad_w = h - w

        left = pad_w // 2
        right = pad_w - left
        
        top = bottom = 0
    else:
        top = bottom = left = right = 0

    padded_image = cv2.copyMakeBorder(curr_image, top, bottom, left, right, cv2.BORDER_CONSTANT, 0)
    padded_image = cv2.resize(padded_image, (28, 28), interpolation=cv2.INTER_CUBIC)

    padded_image = cv2.erode(padded_image, None, iterations=1)
    padded_image = cv2.erode(padded_image, None, )

    images[i] = harsh_gamma_enhance(padded_image) / 255

    print(harsh_gamma_enhance(padded_image) / 255)

print(images)

In [ ]:
from pymep.realParser import parse

MODEL_PATH = "../conv_neural_network/model/penman_cnn.onnx"

session = ort.InferenceSession(MODEL_PATH)

label_decoding = {
    10: '(',
    11: ')',
    12: '+',
    13: '-',
    14: '=',
    15: 'fwd_slash',
    16: 'times'
}

output = session.run(None, {"X": images.astype(np.float32)})[0]
raw_values = list(np.argmax(output, axis=1))
str_values = [str(x) if x < 10 else label_decoding[x] for x in raw_values]
expression = "".join(str_values)

print(parse(expression))

# for image in images:
#     new_image = image.reshape(1, 1, 28, 28).astype(np.float32)

#     output = session.run(None, {"X": new_image.astype(np.float32)})

#     val = np.argmax(output)

#     print(val if val < 10 else label_decoding[val])
#     print(output)
#     print(new_image)
#     print()

# output = session.run(None, {"X": images.astype(np.float32)})

# print(images[0].reshape(1, 1, 28, 28))

# print(output)

In [ ]:
def process_image(image: Image):
    pipeline =              [cv2.erode, harsh_gamma_enhance, normalize]
    pipeline_default_args = [
        {"kernel": None, "iterations": 1},
        {},
        {}
    ]

    # Resizes and inverts image so the larger dimension is 28 and is the proper format for the CNN.
    resized_image = resize_image(image)
    contour_boxes = find_contours(resized_image)

    image_arr = stretch_image(resized_image, contour_boxes)

    for i, image in enumerate(image_arr):
        for j in range(len(pipeline)):
            image = pipeline[j](image, **pipeline_default_args[j])

        image_arr[i] = image

    return image_arr

with open("data/3+3.txt", "r") as file:
    image = Image.open(io.BytesIO(base64.decodebytes(bytes(file.read(), "utf-8"))))

print(process_image(image).shape)